In [4]:
# ===== Setup =====
import json
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

from spark_core import *

dev = pd.read_csv("../data/processed/v2_math_dev.csv")
y = dev["risk"]
is_high = (y == "High").astype(int).values

In [5]:
# ===== Tune the two additional models =====
tuning_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=100)
extra_params = {"stage1": {}, "stage2": {}}
rows = []

for stage, cols in STAGE_COLS.items():
    for name, grid in EXTRA_TUNING_GRIDS.items():
        search = GridSearchCV(make_pipeline(name), grid, scoring=high_avg_precision,
                              cv=tuning_cv, n_jobs=-1)
        search.fit(dev[cols], y)
        extra_params[stage][name] = search.best_params_

        pipe = make_pipeline(name, **search.best_params_)
        runs = oof_high_proba(pipe, dev[cols], y, n_repeats=5)
        rows.append({
            "stage": stage, "model": name,
            "tuning_avg_precision": round(search.best_score_, 3),
            "fresh_avg_precision": round(np.mean([average_precision_score(is_high, r) for r in runs]), 3),
            "precision_at_90_recall": round(np.mean([precision_at_recall(is_high, r) for r in runs]), 3),
        })

with open("../outputs/tables/v2_extra_params.json", "w") as f:
    json.dump(extra_params, f, indent=2)

pd.DataFrame(rows)

c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_bas

,stage,model,tuning_avg_precision,fresh_avg_precision,precision_at_90_recall
0,stage1,SVM,0.817,0.715,0.461
1,stage1,XGBoost,0.809,0.797,0.659
2,stage2,SVM,0.942,0.944,0.855
3,stage2,XGBoost,0.948,0.949,0.854


In [6]:
# ===== Frozen bundles for the new models =====
new_bundles = {}
for name in EXTRA_TUNING_GRIDS:
    estimators = {stage: make_pipeline(name, **extra_params[stage][name]) for stage in STAGE_COLS}
    new_bundles[name] = fit_bundle(estimators, dev)
    save_bundle(new_bundles[name], f"../models/v2_bundle_{name.replace(' ', '_')}.joblib")

print({name: {s: round(t, 3) for s, t in b["thresholds"].items()} for name, b in new_bundles.items()})

c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Asus\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_bas

{'SVM': {'stage1': 0.368, 'stage2': 0.467}, 'XGBoost': {'stage1': 0.429, 'stage2': 0.481}}


In [7]:
# ===== Six models, full metric set =====
holdout = pd.read_csv("../data/processed/v2_math_holdout_do_not_open.csv")
por_full, por_clean = load_portuguese()
test_sets = {"Mathematics hold out": holdout, "Portuguese, no overlap": por_clean}

original = ["Decision Tree", "Random Forest", "KNN", "Logistic regression"]
all_bundles = {name: load_bundle(f"../models/v2_bundle_{name.replace(' ', '_')}.joblib")
               for name in original}
all_bundles.update(new_bundles)

rows = []
for label, data in test_sets.items():
    for name, bundle in all_bundles.items():
        for stage, cols in STAGE_COLS.items():
            pred = score_students(data[cols], bundle)["risk"].values
            proba = stage_probabilities(bundle, data, stage)
            rows.append({"test set": label, "model": name, "stage": stage,
                         **full_metrics(data["risk"].values, pred, proba)})

six_models = pd.DataFrame(rows).round(3)
six_models.to_csv("../outputs/tables/v2_six_models_full_metrics.csv", index=False)
six_models

,test set,model,stage,recall_high,precision_high,high_predicted_low,accuracy,precision_macro,recall_macro,f1_macro,roc_auc_ovr
0,Mathematics hold out,Decision Tree,stage1,0.923,0.632,0,0.696,0.729,0.713,0.705,0.890
1,Mathematics hold out,Decision Tree,stage2,0.923,0.800,0,0.861,0.871,0.870,0.869,0.959
2,Mathematics hold out,Random Forest,stage1,0.923,0.632,0,0.696,0.729,0.713,0.705,0.894
3,Mathematics hold out,Random Forest,stage2,0.923,0.828,0,0.861,0.865,0.870,0.866,0.962
4,Mathematics hold out,KNN,stage1,0.808,0.750,0,0.747,0.770,0.745,0.754,0.889
5,Mathematics hold out,KNN,stage2,0.885,0.852,0,0.848,0.855,0.851,0.853,0.964
6,Mathematics hold out,Logistic regression,stage1,0.808,0.750,0,0.747,0.747,0.771,0.754,0.898
7,Mathematics hold out,Logistic regression,stage2,0.885,0.852,0,0.861,0.865,0.868,0.866,0.966
8,Mathematics hold out,SVM,stage1,0.923,0.632,0,0.544,0.365,0.500,0.421,0.684
9,Mathematics hold out,SVM,stage2,0.923,0.800,0,0.861,0.871,0.870,0.869,0.962
